In [4]:
import os
import ROOT
import pandas as pd
import numpy as np

# =====================================================
# WORKING DIRECTORY AND SETTINGS
# =====================================================
working_directory = "/root/geant4/detector/Lung_ICRP"
os.chdir(working_directory)

file_path = "lung_TissueTumor_430.root"
tree_name = "t"

VOL_TUMOR       = 3
PRO_COMPTON     = 2013
PDG_GAMMA       = 22
selected_volumes = [2, 3,430]   # downstream volumes to report

excel_file = "lung_TissueTumor_430_2.xlsx"

root_file = ROOT.TFile.Open(file_path)
if not root_file or root_file.IsZombie():
    raise OSError(f"Could not open {os.path.abspath(file_path)}")

tree = root_file.Get(tree_name)
if not tree:
    root_file.Close()
    raise KeyError(f"TTree '{tree_name}' not found")

print("Total TTree entries:", tree.GetEntries())

def safe_value(vector, index, default=np.nan):
    return vector[index] if index < len(vector) else default

scatter_rows = []
detector_rows = []

# =====================================================
# SINGLE PASS PER EVENT: find tumor Compton scatters,
# then find each qualifying track's FIRST later step in
# each of the selected downstream volumes.
# =====================================================
for tree_entry, event in enumerate(tree):

    n_records = len(event.pdg)

    # --- Pass A: locate Compton-scatter steps in the tumor for this event ---
    # key: trk -> (comptonStep, k_at_compton, x,y,z,px,py,pz at that step)
    tumor_scatters = {}
    for i in range(n_records):
        if int(event.pdg[i]) != PDG_GAMMA:
            continue
        if int(event.vlm[i]) != VOL_TUMOR:
            continue
        if int(safe_value(event.pro, i, -1)) != PRO_COMPTON:
            continue

        trk = int(safe_value(event.trk, i, -1))
        stp = int(safe_value(event.stp, i, -1))

        # keep the FIRST tumor Compton scatter per track (lowest stp)
        if trk not in tumor_scatters or stp < tumor_scatters[trk]["stp"]:
            tumor_scatters[trk] = {
                "stp": stp,
                "k": float(safe_value(event.k, i)),
                "et": float(safe_value(event.et, i)),
                "x": float(safe_value(event.x, i)),
                "y": float(safe_value(event.y, i)),
                "z": float(safe_value(event.z, i)),
                "px": float(safe_value(event.px, i)),
                "py": float(safe_value(event.py, i)),
                "pz": float(safe_value(event.pz, i)),
            }

    if not tumor_scatters:
        continue   # nothing in this event scattered in the tumor -- skip

    for trk, sc in tumor_scatters.items():
        scatter_rows.append({
            "tree_entry": tree_entry, "trk": trk,
            "scatter_stp": sc["stp"], "scatter_k": sc["k"], "scatter_et": sc["et"],
            "scatter_x": sc["x"], "scatter_y": sc["y"], "scatter_z": sc["z"],
            "scatter_px": sc["px"], "scatter_py": sc["py"], "scatter_pz": sc["pz"],
        })

    # --- Pass B: for each qualifying track, find FIRST step in each
    #     downstream volume, at/after the tumor scatter step ---
    best_hit = {}   # (trk, vlm) -> row dict, keeping min stp
    for i in range(n_records):
        if int(event.pdg[i]) != PDG_GAMMA:
            continue

        trk = int(safe_value(event.trk, i, -1))
        if trk not in tumor_scatters:
            continue   # only tracks that scattered in the tumor

        vol = int(event.vlm[i])
        if vol not in selected_volumes:
            continue

        stp = int(safe_value(event.stp, i, -1))
        if stp < tumor_scatters[trk]["stp"]:
            continue   # must be at/after the tumor scatter

        key = (trk, vol)
        if key not in best_hit or stp < best_hit[key]["stp"]:
            best_hit[key] = {
                "tree_entry": tree_entry, "trk": trk, "vlm": vol,
                "pro": int(safe_value(event.pro, i, -1)),
                "stp": stp,
                "k": float(safe_value(event.k, i)),
                "et": float(safe_value(event.et, i)),
                "de": float(safe_value(event.de, i)),
                "x": float(safe_value(event.x, i)),
                "y": float(safe_value(event.y, i)),
                "z": float(safe_value(event.z, i)),
                "px": float(safe_value(event.px, i)),
                "py": float(safe_value(event.py, i)),
                "pz": float(safe_value(event.pz, i)),
            }

    detector_rows.extend(best_hit.values())

root_file.Close()

scatter_df  = pd.DataFrame(scatter_rows)
detector_df = pd.DataFrame(detector_rows)

print(f"\nQualifying tumor Compton-scatter tracks: {len(scatter_df):,}")
print(f"First-arrival rows across all downstream volumes: {len(detector_df):,}")

# =====================================================
# MERGE: one row per (tree_entry, trk, downstream volume)
# =====================================================
merged = detector_df.merge(scatter_df, on=["tree_entry", "trk"], how="left",
                            suffixes=("", "_scatter"))

with pd.ExcelWriter(excel_file, engine="openpyxl") as writer:
    merged.to_excel(writer, index=False, sheet_name="Scatter_to_Detectors")
    for vol in selected_volumes:
        merged[merged["vlm"] == vol].to_excel(
            writer, index=False, sheet_name=f"Volume_{vol}"
        )

print("\nSaved:", os.path.abspath(excel_file))

Total TTree entries: 100000

Qualifying tumor Compton-scatter tracks: 4,435
First-arrival rows across all downstream volumes: 8,997

Saved: /root/geant4/detector/Lung_ICRP/lung_TissueTumor_430_2.xlsx


In [2]:
sub = df[(df["n"] == 12) & (df["trk"] == 1)].sort_values("stp")
print(sub[["stp", "vlm", "pro", "pdg", "k", "et"]].to_string())

       stp  vlm   pro  pdg           k          et
7723     0    1  1000   22  662.000000  369.856939
8129     0    1  1000   22  662.000000  335.273130
8103     0    1  1000   22  662.000000  400.404461
8088     0    1  1000   22  662.000000  183.952331
7970     0    1  1000   22  662.000000  423.271590
7946     0    1  1000   22  662.000000  115.064775
7689     0    1  1000   22  662.000000  473.902084
7622     0    1  1000   22  662.000000  435.911614
7485     0    1  1000   22  662.000000  312.854398
7435     0    1  1000   22  662.000000  377.081395
8518     0    1  1000   22  662.000000  323.840438
8507     0    1  1000   22  662.000000  429.752731
8371     0    1  1000   22  662.000000  101.009094
8305     0    1  1000   22  662.000000  464.698388
8150     0    1  1000   22  662.000000  392.541741
8904     0    1  1000   22  662.000000   21.684719
29065    0    1  1000   22  662.000000  411.344199
29050    0    1  1000   22  662.000000  451.475882
29557    0    1  1000   22  662